# CSET485 – AI and Society | Assignment #1
## German Credit Data: predictive-model stress testing and fairness analysis

**Student:** Manu Bhardwaj  
**Enrollment:** S24BCAU0034  
**Programme:** BCA  
**Dataset:** UCI Statlog German Credit Data  
**Split:** 70% train / 30% test  
**random_state:** `34` (last three enrolment digits `034`)

This notebook implements the assignment in the instructor's handout. The UCI target is `1=Good`, `2=Bad`; as required, the analysis maps Good to approval=1 and Bad to approval=0. This is a modelling proxy, not an actual record of bank approval decisions.

Run all cells from the repository root after installing `requirements.txt`.

## 0. Run the complete reproducible analysis

The analysis script contains preprocessing, OLS, logistic regression, all four stress tests, tables, and chart generation. Numeric scaling and categorical levels are learned from the training set only. IQR-flagged records are reviewed but retained.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
from cset485_analysis import run_analysis, OUTPUT_DIR

results = run_analysis()
print('Generated:', sorted(p.name for p in OUTPUT_DIR.iterdir()))

## 1. Part 1 – Model comparison

OLS predictions are thresholded at their test-set median; logistic probabilities are thresholded at 0.50. The matrix compares the two prediction rules, not predictions against the true outcome.

In [ ]:
print(json.dumps({'dataset': results['dataset'], 'part1': results['part1']}, indent=2))

## 2.1 Simpson's Paradox Hunt / subgroup analysis

The same seven numeric predictors are used in the full, male and female logistic models. The focus is the standardised credit_amount coefficient and its approximate Wald interval.

In [ ]:
display(pd.read_csv(OUTPUT_DIR / 'subgroup_coefficients.csv'))

## 2.2 Omitted-variable bias

The reduced model removes checking_status and compares four remaining predictors.

In [ ]:
display(pd.read_csv(OUTPUT_DIR / 'omitted_variable_coefficients.csv'))

## 2.3 Adversarial subset construction

Errors are defined as probability >= 0.75 with actual Bad, or probability <= 0.25 with actual Good. All cases are retained when fewer than 50 are found.

In [ ]:
errors = pd.read_csv(OUTPUT_DIR / 'adversarial_high_confidence_errors.csv')
print('High-confidence error rows:', len(errors))
display(errors)

## 2.4 Fairness trade-off analysis

The sensitive variable is personal_status_sex grouped as Male (A91/A93/A94) and Female (A92/A95). EO follows the assignment's specified minimum of TPR ratio and FPR ratio; both components are separately stored in the output CSV.

In [ ]:
display(pd.read_csv(OUTPUT_DIR / 'fairness_threshold_metrics.csv').round(4))
Image(filename=str(OUTPUT_DIR / 'fairness_tradeoff_chart.png'))

## 3. Preprocessing and outlier audit

No missing values are reported by UCI. Thirteen categorical columns are one-hot encoded; seven numeric/integer columns are standardised using training data only. IQR flags are reported without deleting records.

In [ ]:
print(json.dumps(results['preprocessing_outlier_audit'], indent=2))

## 4. Interpretation and limitations

The outputs in `outputs/` contain tables, a summary JSON and a fairness chart. This is an analysis of a 1,000-record historical dataset, not an operational credit-scoring system. The assignment's Good/Bad-to-approval mapping is only a proxy for coursework.